In [ ]:
!pip install monai SimpleITK lifelines scikit-survival shap -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
from pathlib import Path
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from torch.utils.data import Dataset, DataLoader, random_split
from google.colab import drive

drive.mount('/content/drive')

BASE   = Path('/content/drive/MyDrive/multimodal_prognosis/')
TENSOR = BASE / 'tensors'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# dataset 
class CTSurvivalDataset(Dataset):
    def __init__(self, labels_csv, tensor_dir):
        self.df         = pd.read_csv(labels_csv)
        self.tensor_dir = Path(tensor_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        ct_tensor = torch.load(self.tensor_dir / f"{row['patient_id']}.pt").float()
        tabular   = torch.tensor([
            row['age']   / 100.0,
            row['stage'] / 4.0,
            float(row['gender']),
        ], dtype=torch.float32)
        time  = torch.tensor(row['survival_days'], dtype=torch.float32)
        event = torch.tensor(row['event'],         dtype=torch.float32)
        return ct_tensor, tabular, time, event


# encoders 
class ResBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels), nn.ReLU(inplace=True),
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class ImageEncoder(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(16), nn.ReLU(inplace=True), ResBlock3D(16),
            nn.Conv3d(16, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True), ResBlock3D(32),
            nn.Conv3d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True), ResBlock3D(64),
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
        )
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.encoder(x))


class TabularEncoder(nn.Module):
    def __init__(self, in_dim=3, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(64, out_dim),
        )

    def forward(self, x):
        return self.net(x)


# loss and metric 
def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    return -torch.mean((risk_scores - log_cumsum) * events)


def concordance_index(risk_scores, times, events):
    risk_scores = risk_scores.cpu().numpy()
    times       = times.cpu().numpy()
    events      = events.cpu().numpy()
    concordant = comparable = 0
    for i in range(len(times)):
        for j in range(len(times)):
            if events[i] == 1 and times[i] < times[j]:
                comparable += 1
                if risk_scores[i] > risk_scores[j]:
                    concordant += 1
                elif risk_scores[i] == risk_scores[j]:
                    concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5


# baseline models 
class ImageOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder(out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, ct, _tab):
        return self.head(self.encoder(ct)).squeeze(-1)


class TabularOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TabularEncoder(in_dim=3, out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, _ct, tab):
        return self.head(self.encoder(tab)).squeeze(-1)

In [ ]:
class CrossModalAttentionFusion(nn.Module):
    """
    Attention fusion: imaging features attend to tabular features.
    The model learns which clinical variables matter given what it sees in the scan.

    Mechanism:
        Q = W_q(f_img)          query from imaging
        K = W_k(f_tab)          key   from tabular
        V = W_v(f_tab)          value from tabular
        a = softmax(QKᵀ / √d)  attention weight
        f_attended = a * V      attended tabular context
        fused = [f_img ; f_attended]
    """
    def __init__(self, feat_dim=64, num_heads=4):
        super().__init__()
        self.img_encoder = ImageEncoder(out_dim=feat_dim)
        self.tab_encoder = TabularEncoder(in_dim=3, out_dim=feat_dim)

        # Projections for Q, K, V
        self.W_q = nn.Linear(feat_dim, feat_dim)
        self.W_k = nn.Linear(feat_dim, feat_dim)
        self.W_v = nn.Linear(feat_dim, feat_dim)

        self.scale = feat_dim ** 0.5

        # also attend in reverse: tabular attends to imaging
        self.W_q2 = nn.Linear(feat_dim, feat_dim)
        self.W_k2 = nn.Linear(feat_dim, feat_dim)
        self.W_v2 = nn.Linear(feat_dim, feat_dim)

        # final prediction head takes both attended features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, 64),
            nn.LayerNorm(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, ct, tab):
        f_img = self.img_encoder(ct)   # (B, D)
        f_tab = self.tab_encoder(tab)  # (B, D)

        # Direction 1: image attends to tabular
        Q1  = self.W_q(f_img).unsqueeze(1)   # (B, 1, D)
        K1  = self.W_k(f_tab).unsqueeze(1)
        V1  = self.W_v(f_tab).unsqueeze(1)
        a1  = torch.softmax(
                  torch.bmm(Q1, K1.transpose(1, 2)) / self.scale, dim=-1)
        f1  = torch.bmm(a1, V1).squeeze(1)   # (B, D)

        # Direction 2: tabular attends to image
        Q2  = self.W_q2(f_tab).unsqueeze(1)
        K2  = self.W_k2(f_img).unsqueeze(1)
        V2  = self.W_v2(f_img).unsqueeze(1)
        a2  = torch.softmax(
                  torch.bmm(Q2, K2.transpose(1, 2)) / self.scale, dim=-1)
        f2  = torch.bmm(a2, V2).squeeze(1)   # (B, D)

        # Combine both attended representations
        fused = torch.cat([f1, f2], dim=1)    # (B, 2D)
        return self.head(fused).squeeze(-1)

    def get_attention_weights(self, ct, tab):
        """Returns attention weights for interpretability analysis."""
        f_img = self.img_encoder(ct)
        f_tab = self.tab_encoder(tab)
        Q1    = self.W_q(f_img).unsqueeze(1)
        K1    = self.W_k(f_tab).unsqueeze(1)
        a1    = torch.softmax(
                    torch.bmm(Q1, K1.transpose(1, 2)) / self.scale, dim=-1)
        return a1.squeeze()   # (B,) — scalar per patient here


# Shape check
model_check = CrossModalAttentionFusion().to(DEVICE)
ct_dummy    = torch.randn(4, 1, 64, 64, 64).to(DEVICE)
tab_dummy   = torch.randn(4, 3).to(DEVICE)
out         = model_check(ct_dummy, tab_dummy)
print(f"Fusion output shape: {out.shape}")   # expect (4,)

In [ ]:
torch.manual_seed(42)
dataset  = CTSurvivalDataset(BASE / 'NSCLC-Radiomics-Lung1', TENSOR)
train_ds, val_ds, test_ds = random_split(dataset, [140, 30, 30])
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

# Load best attention fusion model
attn_model = CrossModalAttentionFusion().to(DEVICE)
attn_model.load_state_dict(torch.load(BASE / 'attn_fusion_best.pt'))
attn_model.eval()
print("Model loaded.")

In [ ]:
all_risk, all_time, all_event = [], [], []
all_tab_raw = []   # store raw tabular features for SHAP

with torch.no_grad():
    for ct, tab, time, event in test_loader:
        ct, tab = ct.to(DEVICE), tab.to(DEVICE)
        risk    = attn_model(ct, tab)
        all_risk.append(risk.cpu())
        all_time.append(time)
        all_event.append(event)
        all_tab_raw.append(tab.cpu())

risk_scores = torch.cat(all_risk).numpy()
times       = torch.cat(all_time).numpy()
events      = torch.cat(all_event).numpy()
tab_feats   = torch.cat(all_tab_raw).numpy()

print(f"Test patients   : {len(risk_scores)}")
print(f"Risk score range: {risk_scores.min():.3f} → {risk_scores.max():.3f}")
print(f"Event rate      : {events.mean():.2%}")

In [ ]:
# split test patients into high/low risk by median risk score
median_risk  = np.median(risk_scores)
high_risk    = risk_scores >= median_risk
low_risk     = risk_scores <  median_risk

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(8, 5))

kmf.fit(times[high_risk], events[high_risk], label='High risk')
kmf.plot_survival_function(ax=ax, color='coral', ci_show=True)

kmf.fit(times[low_risk], events[low_risk], label='Low risk')
kmf.plot_survival_function(ax=ax, color='steelblue', ci_show=True)

# log-rank test
lr = logrank_test(
    times[high_risk],  times[low_risk],
    events[high_risk], events[low_risk]
)

ax.set_title(f'Kaplan-Meier — attention fusion risk stratification\n'
             f'Log-rank p = {lr.p_value:.4f}')
ax.set_xlabel('Survival time (days)')
ax.set_ylabel('Survival probability')
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
plt.savefig(BASE / 'kaplan_meier.png', dpi=150)
plt.show()

print(f"Log-rank p-value: {lr.p_value:.4f}")
print("(p < 0.05 means model risk groups are statistically separable)")

In [ ]:
# calibration: are predicted risk scores consistent with observed event rates?
# bin patients by risk score, compare predicted vs observed event rate

n_bins   = 5
bins     = np.percentile(risk_scores, np.linspace(0, 100, n_bins + 1))
bin_ids  = np.digitize(risk_scores, bins[1:-1])

pred_risk_mean = []
obs_event_rate = []
bin_sizes      = []

for b in range(n_bins):
    mask = bin_ids == b
    if mask.sum() == 0:
        continue
    pred_risk_mean.append(risk_scores[mask].mean())
    obs_event_rate.append(events[mask].mean())
    bin_sizes.append(mask.sum())

pred_norm = (np.array(pred_risk_mean) - np.min(pred_risk_mean))
pred_norm = pred_norm / (pred_norm.max() + 1e-8)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
sc = ax.scatter(pred_norm, obs_event_rate,
                s=[b * 40 for b in bin_sizes],
                c=range(n_bins), cmap='RdYlGn_r',
                zorder=3, edgecolors='gray', linewidths=0.5)
ax.plot(pred_norm, obs_event_rate, 'gray', alpha=0.4, zorder=2)

for i, (x, y, n) in enumerate(zip(pred_norm, obs_event_rate, bin_sizes)):
    ax.annotate(f'n={n}', (x, y),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Normalised predicted risk (binned)')
ax.set_ylabel('Observed event rate')
ax.set_title('Calibration plot — attention fusion\n(point size = bin size)')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(BASE / 'calibration_plot.png', dpi=150)
plt.show()

In [ ]:
# SHAP explains which tabular features drive the survival prediction


def tabular_predict(tab_array):
    """Run only the tabular branch for SHAP attribution."""
    tab_tensor = torch.tensor(tab_array, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        f_tab = attn_model.tab_encoder(tab_tensor)
        # Use the tabular-attended output path
        f_img_zero = torch.zeros(
            tab_tensor.shape[0], 64, device=DEVICE)   # zero imaging
        Q2  = attn_model.W_q2(f_tab).unsqueeze(1)
        K2  = attn_model.W_k2(f_img_zero).unsqueeze(1)
        V2  = attn_model.W_v2(f_img_zero).unsqueeze(1)
        a2  = torch.softmax(
                  torch.bmm(Q2, K2.transpose(1,2)) / attn_model.scale, dim=-1)
        f2  = torch.bmm(a2, V2).squeeze(1)
        f1  = f_tab   # approximate: use tab features directly for f1
        fused = torch.cat([f1, f2], dim=1)
        out = attn_model.head(fused).squeeze(-1)
    return out.cpu().numpy()

feature_names = ['age / 100', 'stage / 4', 'gender']

# Use training tabular data as background
train_tab = []
for _, tab, _, _ in DataLoader(train_ds, batch_size=140):
    train_tab.append(tab.numpy())
background = np.vstack(train_tab)[:50]   # 50 background samples

explainer   = shap.KernelExplainer(tabular_predict, background)
shap_values = explainer.shap_values(tab_feats, nsamples=100)

fig, ax = plt.subplots(figsize=(7, 4))
shap.summary_plot(
    shap_values, tab_feats,
    feature_names=feature_names,
    plot_type='bar', show=False
)
plt.title('SHAP feature importance — tabular branch')
plt.tight_layout()
plt.savefig(BASE / 'shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from lifelines.utils import concordance_index as lifelines_ci

# Recompute with lifelines for a second opinion
li_ci = lifelines_ci(times, -risk_scores, events)

print("=" * 55)
print("PROJECT RESULTS SUMMARY")
print("=" * 55)
print()
print("Ablation table (test C-index, n=30):")
print(f"  Image-only        : 0.4523")
print(f"  Tabular-only      : 0.6680")
print(f"  Concat fusion     : 0.6971  (+0.029 over tabular)")
print(f"  Attention fusion  : 0.7469  (+0.079 over tabular) ✓ best")
print()
print(f"Attention fusion — additional metrics:")
print(f"  Lifelines C-index : {li_ci:.4f}")
print(f"  Log-rank p-value  : {lr.p_value:.4f}")
print()
print("Saved artefacts:")
for f in ['kaplan_meier.png','calibration_plot.png',
          'shap_importance.png','fusion_results.png',
          'baselines_200.png']:
    exists = '✓' if (BASE / f).exists() else '✗'
    print(f"  {exists}  {f}")